In [17]:
import json
import os
from pathlib import Path
import urllib.request

import openai


def load_env_file(path: str = ".env"):
    env_path = Path(path)
    if not env_path.exists():
        return

    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file()

MOVIE_API_BASE_URL = os.getenv(
    "MOVIE_API_BASE_URL", "https://nomad-movies.nomadcoders.workers.dev"
)

client = openai.OpenAI()
messages = []
_tool_turn_buffer: list = []
excluded_movie_ids: set[str] = set()

In [18]:
def _fetch_movie_api(path: str):
    url = f"{MOVIE_API_BASE_URL}{path}"
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "Mozilla/5.0",
        },
    )
    with urllib.request.urlopen(request, timeout=10) as response:
        return json.loads(response.read().decode("utf-8"))


def get_popular_movies():
    movies = _fetch_movie_api("/movies")
    filtered = [m for m in movies if str(m.get("id")) not in excluded_movie_ids]
    return json.dumps(filtered)


def register_excluded_movies(movie_ids: list):
    for mid in movie_ids:
        excluded_movie_ids.add(str(mid))
    return json.dumps({"ok": True, "total_excluded": len(excluded_movie_ids)})


def get_movie_details(id):
    details = _fetch_movie_api(f"/movies/{id}")
    return json.dumps(details)


def get_movie_credits(id):
    credits = _fetch_movie_api(f"/movies/{id}/credits")
    return json.dumps(credits)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "register_excluded_movies": register_excluded_movies,
}

In [19]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies", # function: get_popular_movies()
            "description": "Get the most popular movie.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details", # function: get_movie_details(id)
            "description": "Get details for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits", # function: credits(id)
            "description": "Get credit score for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "register_excluded_movies",
            "description": (
                "Record TMDB movie ids that must not be suggested again: any title you recommended "
                "in this chat, and any film the user says they already watched. Call with ids from "
                "API responses (same turn as recommendations when possible)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_ids": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "TMDB movie ids as strings.",
                    }
                },
                "required": ["movie_ids"],
            },
        },
    },
]

In [20]:
from openai.types.chat import ChatCompletionMessage

BASE_SYSTEM = (
    "You are a helpful movie assistant. "
    "Whenever you recommend movies, call register_excluded_movies with each title's TMDB id "
    "from the API data (in the same assistant turn as the recommendation). "
    "When the user says they already watched a movie, look up its id from earlier tool data or "
    "fetch details, then call register_excluded_movies. "
    "Do not recommend movies whose ids appear in the exclusion list below.\n\n"
)


def messages_for_chat_completion():
    if excluded_movie_ids:
        try:
            ordered = sorted(excluded_movie_ids, key=int)
        except ValueError:
            ordered = sorted(excluded_movie_ids)
        suffix = (
            "Excluded TMDB ids (already recommended or user already saw): "
            + ", ".join(ordered)
            + "."
        )
    else:
        suffix = "No movies excluded yet."
    return [{"role": "system", "content": BASE_SYSTEM + suffix}] + messages + _tool_turn_buffer


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        _tool_turn_buffer.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            # print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            if function_to_run is None:
                result = f"Error: unknown function '{function_name}'."
                print(result)
            else:
                try:
                    result = function_to_run(**arguments)
                except Exception as e:
                    result = f"Error running {function_name}: {e}"
                    print(result)

            # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            _tool_turn_buffer.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": str(result),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        _tool_turn_buffer.clear()
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for_chat_completion(),
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [22]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        _tool_turn_buffer.clear()
        print(f"User: {message}")
        call_ai()

User: I like SF movie. I already watched Interstellar. Can you recommend me another SF movie?
AI: Since you've already watched *Interstellar*, I recommend checking out *Project Hail Mary*. It’s an intriguing science fiction film about a science teacher who wakes up on a spaceship with no recollection of who he is and his mission to solve a riddle that could save Earth from extinction.

![Project Hail Mary](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)

Here are some details:
- **Release Date**: March 15, 2026
- **Vote Average**: 8.2
- **Overview**: Ryland Grace wakes up on a spaceship light years from home, tasked with solving a mysterious substance causing the sun to die out.

I’m going to register *Interstellar* as an excluded movie. 
User: I heard about Project Hail Mary. It's SF but comedy as well. Right?
AI: Yes, *Project Hail Mary* has comedic elements that blend well with its science fiction plot. The story includes light-hearted moments alongside the more ser

In [23]:
messages

[{'role': 'user',
  'content': 'I like SF movie. I already watched Interstellar. Can you recommend me another SF movie?'},
 {'role': 'assistant',
  'content': "Since you've already watched *Interstellar*, I recommend checking out *Project Hail Mary*. It’s an intriguing science fiction film about a science teacher who wakes up on a spaceship with no recollection of who he is and his mission to solve a riddle that could save Earth from extinction.\n\n![Project Hail Mary](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)\n\nHere are some details:\n- **Release Date**: March 15, 2026\n- **Vote Average**: 8.2\n- **Overview**: Ryland Grace wakes up on a spaceship light years from home, tasked with solving a mysterious substance causing the sun to die out.\n\nI’m going to register *Interstellar* as an excluded movie. "},
 {'role': 'user',
  'content': "I heard about Project Hail Mary. It's SF but comedy as well. Right?"},
 {'role': 'assistant',
  'content': "Yes, *Project Hail M